# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [ ]:
from datetime import datetime
from agent import Agent
from ragas.integrations.langgraph import convert_to_ragas_messages
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import ToolCall
from ragas.metrics import AgentGoalAccuracyWithReference
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

from ragas.messages import HumanMessage, AIMessage
from ragas.metrics import ToolCallAccuracy



/tmp/ipykernel_197599/1132309893.py:6: DeprecationWarning: Importing AgentGoalAccuracyWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AgentGoalAccuracyWithReference
  from ragas.metrics import AgentGoalAccuracyWithReference


In [2]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
You are an energy efficiency advisor helping customers reduce their energy consumption and bills.

## Role and scope

Your job is to:
- Diagnose where a customer's energy is being wasted
- Give ranked, actionable recommendations with realistic savings estimates

You are not a customer service agent. Do not handle billing disputes, outages, or account changes — direct those to the customer's supplier.

## Behaviour

- Ask targeted questions before recommending. Don't ask all at once — prioritise the one most likely to unlock the most useful advice.
- Rank recommendations by effort-to-impact ratio.
- Be honest about uncertainty. If you don't have enough information to give a reliable estimate, say so and ask for what you need.

## Communication style

- Direct and plain. No filler phrases ("Great question!", "Absolutely!").
- Short answers unless complexity demands otherwise.
- Define technical terms inline in plain language — don't assume familiarity with kWh, U-values, load profiles, or tariff structures.

## Knowledge boundaries

Only use information contained in your current context, you will be provided tools to query against known data.

## Safety

If a customer describes something that sounds like a gas leak, electrical fault, or carbon monoxide risk, stop the energy advice and tell them to contact emergency services or a certified engineer immediately.
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric car tomorrow, consider the following:

### Best Charging Times
1. **Early Morning (12 AM - 5 AM)**: 
   - Rates are low (around $0.08 per kWh).
   - No solar power available.

2. **Mid-Morning to Early Afternoon (6 AM - 2 PM)**:
   - Solar power starts around 6 AM, peaking around 12 PM to 2 PM.
   - Rates increase after 6 AM but are still manageable.
   - Best time to charge is between **6 AM and 10 AM** when solar generation starts to kick in.

3. **Late Afternoon (3 PM - 5 PM)**:
   - Solar power is still available, but rates are higher (up to $0.16 per kWh).
   - Charging during this time can still be beneficial if you have solar generation.

### Recommendation
- **Charge between 6 AM and 10 AM** to take advantage of both low rates and the beginning of solar power generation. If you can wait until around **12 PM to 2 PM**, that would be ideal for maximizing solar energy, but rates will be higher.

### Summary
- *

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_electricity_prices
- get_weather_forecast


## 2. Define Test Cases

In [ ]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [8]:
test_cases = [
    {
        "id": "ev_charging",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "thermostat_price_spike",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain a temperature recommendation, reasoning tied to the price spike window, and consideration of outdoor weather conditions",
    },
    {
        "id": "reduce_usage_tips",
        "question": "Suggest three ways I can reduce energy use based on my usage history.",
        "expected_tools": ["query_energy_usage", "get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "The response should contain three distinct, actionable suggestions grounded in the user's actual usage patterns",
    },
    {
        "id": "dishwasher_off_peak_savings",
        "question": "How much can I save by running my dishwasher during off-peak hours?",
        "expected_tools": ["get_electricity_prices", "query_energy_usage", "calculate_energy_savings"],
        "expected_response": "The response should contain a quantified savings estimate, the peak vs off-peak price differential, and an assumption about dishwasher energy consumption",
    },
    {
        "id": "pool_pump_schedule",
        "question": "What's the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "get_electricity_prices"],
        "expected_response": "The response should contain a day/time recommendation across the week, alignment with solar generation, and consideration of electricity prices",
    },
    {
        "id": "monthly_bill_review",
        "question": "Give me a summary of last month's energy use and tell me where I can cut back.",
        "expected_tools": ["get_recent_energy_summary", "query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should contain a summary of last month's consumption, identification of the highest-use categories or periods, and targeted reduction suggestions",
    },
    {
        "id": "solar_battery_payback",
        "question": "If I added a home battery, how much would I save each year given my solar generation and current usage?",
        "expected_tools": ["query_solar_generation", "query_energy_usage", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should contain an annual savings estimate, the assumed battery capacity, and reasoning that combines solar surplus, usage patterns, and the peak vs off-peak price differential",
    },
    {
        "id": "solar_power_maximisation",
        "question": "How can I get the most out of my solar panels this week — what should I shift to run during peak generation hours?",
        "expected_tools": ["query_solar_generation", "get_weather_forecast", "query_energy_usage", "get_electricity_prices", "search_energy_tips", "calculate_energy_savings"],
        "expected_response": "The response should contain the forecast peak-generation windows for the week, specific loads or appliances to shift into those windows based on the user's usage profile, the value of self-consuming vs exporting given current prices, and an estimated saving",
    },
    {
        "id": "heatwave_prep",
        "question": "There's a heatwave coming this weekend — how should I prepare to keep my bill down?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips", "query_solar_generation"],
        "expected_response": "The response should contain heatwave-specific cooling guidance, awareness of expected price movements during peak demand, and use of solar generation where applicable",
    },
    {
        "id": "laundry_timing",
        "question": "When's the cheapest time to do laundry this week?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "query_solar_generation", "calculate_energy_savings"],
        "expected_response": "The response should contain a specific day/time recommendation, the price differential vs a typical run, and consideration of solar generation if available",
    },
    {
        "id": "usage_anomaly",
        "question": "My bill seems higher than usual — can you figure out what's driving it?",
        "expected_tools": ["get_recent_energy_summary", "query_energy_usage", "search_energy_tips", "calculate_energy_savings"],
        "expected_response": "The response should contain identification of the likely cause (period, appliance category, or behaviour change), comparison against the user's baseline, and suggested corrective actions with rough savings",
    },
]


if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_price_spike
Question: What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
--------------------------------------------------

Test 3: reduce_usage_tips
Question: Suggest three ways I can reduce energy use based on my usage history.
--------------------------------------------------

Test 4: dishwasher_off_peak_savings
Question: How much can I save by running my dishwasher during off-peak hours?
--------------------------------------------------

Test 5: pool_pump_schedule
Question: What's the best time to run my pool pump this week based on the weather forecast?
--------------------------------------------------

Test 6: monthly_bill_review
Question: Give me a summary of last month's energy use and tell me where I can cut back.

In [11]:
test_results

[{'test_id': 'ev_charging',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='15e434e1-ce84-45a9-b2b2-ae1f76cdb3cc'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='8f0b6787-da9d-4ffb-b969-bd2bb33fe653'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 1118, 'total_tokens': 1140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c96d4b6b5b', 'id': 'c

## 4. Evaluate Responses

In [ ]:


# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage
 
llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.0,
    )

In [ ]:
# TODO: Create a response evaluator
async def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""

    sample = MultiTurnSample(
        user_input=[
            HumanMessage(content=question),
            AIMessage(content=final_response),
        ],
        reference=expected_response,
    )

    scorer = AgentGoalAccuracyWithReference()
    scorer.llm = LangchainLLMWrapper(llm)
    
    
    return await scorer.multi_turn_ascore(sample)


In [ ]:
# TODO: Create a tool usage evaluator
async def evaluate_tool_usage(messages, expected_tools_list):
    """Evaluate if the right tools were used"""
    sample = MultiTurnSample(
        user_input=messages,
        reference_tool_calls=expected_tools_list,
    )
    scorer = ToolCallAccuracy()
    scorer.llm = llm
    return await scorer.multi_turn_ascore(sample)

In [ ]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement

# test_results :

# [{'test_id': 'ev_charging',
#   'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
#   'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='15e434e1-ce84-45a9-b2b2-ae1f76cdb3cc'),
#     HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='8f0b6787-da9d-4ffb-b969-bd2bb33fe653'),
#     AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 1118, 'total_tokens': 1140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c96d4b6b5b', 'id': 'chatcmpl-Df5sC7oqSstrxwsPvRTbIW7mOwsqP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='energy_advisor', id='lc_run--019e21f8-8067-73c1-af60-19bcc4a7dcb8-0', tool_calls=[{'name': 'get_electricity_prices', 'args': {'date': '2023-10-07'}, 'id': 'call_n2UErar7QB4S9Yehxcnk8ss7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1118, 'output_tokens': 22, 'total_tokens': 1140, 'input_token_details': {'audio': 0, 'cache_read': 1024}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#     ToolMessage(content='{"date": "2023-10-07", "pricing_type": "time_of_use", "currency": "USD", "unit": "per_kWh", "hourly_rates": [{"hour": 0, "rate": 0.0823, "period": "off_peak", "demand_charge": 0.0}, {"hour": 1, "rate": 0.0791, "period": "off_peak", "demand_charge": 0.0}, {"hour": 2, "rate": 0.0768, "period": "off_peak", "demand_charge": 0.0}, {"hour": 3, "rate": 0.0754, "period": "off_peak", "demand_charge": 0.0}, {"hour": 4, "rate": 0.0782, "period": "off_peak", "demand_charge": 0.0}, {"hour": 5, "rate": 0.0815, "period": "off_peak", "demand_charge": 0.0}, {"hour": 6, "rate": 0.1432, "period": "peak", "demand_charge": 2.45}, {"hour": 7, "rate": 0.1587, "period": "peak", "demand_charge": 2.68}, {"hour": 8, "rate": 0.1623, "period": "peak", "demand_charge": 2.71}, {"hour": 9, "rate": 0.1498, "period": "peak", "demand_charge": 2.52}, {"hour": 10, "rate": 0.1376, "period": "peak", "demand_charge": 2.31}, {"hour": 11, "rate": 0.1412, "period": "peak", "demand_charge": 2.38}, {"hour": 12, "rate": 0.1465, "period": "peak", "demand_charge": 2.47}, {"hour": 13, "rate": 0.1521, "period": "peak", "demand_charge": 2.56}, {"hour": 14, "rate": 0.1554, "period": "peak", "demand_charge": 2.61}, {"hour": 15, "rate": 0.1612, "period": "peak", "demand_charge": 2.69}, {"hour": 16, "rate": 0.1738, "period": "peak", "demand_charge": 2.91}, {"hour": 17, "rate": 0.1856, "period": "peak", "demand_charge": 3.12}, {"hour": 18, "rate": 0.1912, "period": "peak", "demand_charge": 3.24}, {"hour": 19, "rate": 0.1843, "period": "peak", "demand_charge": 3.09}, {"hour": 20, "rate": 0.1687, "period": "peak", "demand_charge": 2.83}, {"hour": 21, "rate": 0.1534, "period": "peak", "demand_charge": 2.58}, {"hour": 22, "rate": 0.0967, "period": "off_peak", "demand_charge": 0.0}, {"hour": 23, "rate": 0.0876, "period": "off_peak", "demand_charge": 0.0}]}', name='get_electricity_prices', id='f8b68e97-6fbb-44bd-bff1-b365f89c25ce', tool_call_id='call_n2UErar7QB4S9Yehxcnk8ss7'),
#     AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1947, 'total_tokens': 1970, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1920}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c96d4b6b5b', 'id': 'chatcmpl-Df5sIgOS1LfvxIN3hXD214KNxuayK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='energy_advisor', id='lc_run--019e21f8-993e-7db2-9c54-f125e648276c-0', tool_calls=[{'name': 'get_weather_forecast', 'args': {'location': 'San Francisco, CA', 'days': 1}, 'id': 'call_uup10l6RHdHHniidRPGtoYqk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1947, 'output_tokens': 23, 'total_tokens': 1970, 'input_token_details': {'audio': 0, 'cache_read': 1920}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#     ToolMessage(content='{"location": "San Francisco, CA", "forecast_days": "1", "current": {"temperature_c": 6, "condition": "cloudy", "humidity": 65, "wind_speed": 56.5}, "hourly": [{"hour": 0, "temperature_c": 22, "condition": "partly_cloudy", "solar_irradiance": 0, "humidity": 55, "wind_speed": 21.8}, {"hour": 1, "temperature_c": 13, "condition": "partly_cloudy", "solar_irradiance": 0, "humidity": 65, "wind_speed": 75.4}, {"hour": 2, "temperature_c": 25, "condition": "sunny", "solar_irradiance": 0, "humidity": 66, "wind_speed": 49.8}, {"hour": 3, "temperature_c": 24, "condition": "partly_cloudy", "solar_irradiance": 0, "humidity": 54, "wind_speed": 64.4}, {"hour": 4, "temperature_c": 14, "condition": "cloudy", "solar_irradiance": 0, "humidity": 80, "wind_speed": 44.9}, {"hour": 5, "temperature_c": 15, "condition": "sunny", "solar_irradiance": 0, "humidity": 71, "wind_speed": 16.3}, {"hour": 6, "temperature_c": 16, "condition": "partly_cloudy", "solar_irradiance": 752.4, "humidity": 74, "wind_speed": 63.0}, {"hour": 7, "temperature_c": 9, "condition": "sunny", "solar_irradiance": 737.5, "humidity": 80, "wind_speed": 65.0}, {"hour": 8, "temperature_c": 18, "condition": "partly_cloudy", "solar_irradiance": 296.9, "humidity": 72, "wind_speed": 3.2}, {"hour": 9, "temperature_c": 20, "condition": "cloudy", "solar_irradiance": 498.3, "humidity": 51, "wind_speed": 45.0}, {"hour": 10, "temperature_c": 10, "condition": "sunny", "solar_irradiance": 217.8, "humidity": 59, "wind_speed": 66.6}, {"hour": 11, "temperature_c": 7, "condition": "sunny", "solar_irradiance": 6.3, "humidity": 73, "wind_speed": 25.1}, {"hour": 12, "temperature_c": 7, "condition": "cloudy", "solar_irradiance": 237.3, "humidity": 50, "wind_speed": 69.0}, {"hour": 13, "temperature_c": 23, "condition": "cloudy", "solar_irradiance": 112.8, "humidity": 57, "wind_speed": 77.3}, {"hour": 14, "temperature_c": 15, "condition": "partly_cloudy", "solar_irradiance": 662.0, "humidity": 78, "wind_speed": 1.1}, {"hour": 15, "temperature_c": 10, "condition": "cloudy", "solar_irradiance": 905.7, "humidity": 78, "wind_speed": 63.3}, {"hour": 16, "temperature_c": 19, "condition": "partly_cloudy", "solar_irradiance": 553.3, "humidity": 58, "wind_speed": 12.6}, {"hour": 17, "temperature_c": 22, "condition": "sunny", "solar_irradiance": 640.5, "humidity": 71, "wind_speed": 31.9}, {"hour": 18, "temperature_c": 12, "condition": "cloudy", "solar_irradiance": 918.5, "humidity": 82, "wind_speed": 14.3}, {"hour": 19, "temperature_c": 21, "condition": "cloudy", "solar_irradiance": 198.5, "humidity": 75, "wind_speed": 71.4}, {"hour": 20, "temperature_c": 9, "condition": "partly_cloudy", "solar_irradiance": 470.8, "humidity": 48, "wind_speed": 2.9}, {"hour": 21, "temperature_c": 21, "condition": "partly_cloudy", "solar_irradiance": 0, "humidity": 50, "wind_speed": 69.9}, {"hour": 22, "temperature_c": 7, "condition": "cloudy", "solar_irradiance": 0, "humidity": 81, "wind_speed": 53.7}, {"hour": 23, "temperature_c": 15, "condition": "sunny", "solar_irradiance": 0, "humidity": 47, "wind_speed": 15.6}]}', name='get_weather_forecast', id='ee419b2c-07a9-4288-a732-b2d9e142867e', tool_call_id='call_uup10l6RHdHHniidRPGtoYqk'),
#     AIMessage(content='To minimize costs and maximize solar power for charging your electric car tomorrow, consider the following:\n\n### Best Charging Times\n1. **Early Morning (12 AM - 5 AM)**: \n   - Rates are low (around $0.08 per kWh).\n   - No solar power available.\n\n2. **Late Morning to Early Afternoon (10 AM - 2 PM)**:\n   - Solar power starts to generate around 6 AM, peaking around 1 PM.\n   - Rates are higher during peak hours (around $0.14 to $0.16 per kWh).\n\n3. **Best Time to Charge**: \n   - **Charge between 12 PM and 2 PM** when solar generation is decent (up to 905.7 W/m²) and rates are still manageable (around $0.14 to $0.15 per kWh).\n\n### Summary\n- **Charge your car between 12 PM and 2 PM** to take advantage of solar power while keeping costs lower than peak evening rates. \n\nIf you have any specific charging needs or preferences, let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 226, 'prompt_tokens': 3165, 'total_tokens': 3391, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e19c763214', 'id': 'chatcmpl-Df5sNWTyUivwhEi9aSC7XtYa7fc2Q', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, name='energy_advisor', id='lc_run--019e21f8-b23c-7e23-a3f6-ee55897bd17a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3165, 'output_tokens': 226, 'total_tokens': 3391, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]},
#   'expected_tools': ['get_weather_forecast', 'get_electricity_prices'],
#   'expected_response': 'The response should contain time recommendation, cost analysis and solar consideration',
#   'timestamp': '2026-05-13T16:33:30.815371'}]

def generate_evaluation_report():
    pass